# Install Requirements

In [1]:
!uv pip install --system \
    docling \
    faiss-cpu \
    ragas==0.1.21 \
    langchain-community==0.2.19 \
    langchain-huggingface \
    langchain-groq \
    langchain-google-vertexai \
    unstructured \
    groq \
    sentence-transformers \
    transformers

Using Python 3.12.13 environment at: /usr
Resolved 218 packages in 1.76s
Prepared 5 packages in 3.48s
Uninstalled 4 packages in 77ms
Installed 5 packages in 23ms
 - google-cloud-storage==3.10.1
 + google-cloud-storage==2.19.0
 + langchain-google-vertexai==1.0.8
 - langchain-groq==1.1.2
 + langchain-groq==0.1.10
 - langchain-huggingface==1.2.2
 + langchain-huggingface==0.0.3
 - opencv-python==4.13.0.92
 + opencv-python==4.11.0.86


# Pipeline Description

> This notebook builds an end-to-end RAG pipeline over the Moroccan concrete standard **NM 10.1.008-2007** (*Beton : Specifications, performances, production et conformite*). The document is a 61-page normative PDF covering concrete classification, exposure classes, conformity control, production control, and evaluation procedures.

> The workflow covers: PDF parsing via Docling, hybrid chunking, sentence-transformer embeddings, FAISS in-memory indexing, Jina reranking, Groq LLM augmentation, and Ragas-based evaluation.

# Document Parsing

> Docling provides a built-in PDF parser with optional OCR and VLM enhancements. We use the default configuration. The NM 10.1.008 PDF has a native text layer so standard extraction is sufficient. **GPU acceleration** is used when available to speed up processing.

In [2]:
from docling.document_converter import DocumentConverter

In [3]:
# Path to the uploaded NM 10.1.008 PDF
# If running on Colab, upload the file first and adjust the path
SOURCE_PATH = "/content/Norme Marocaine du Béton NM 10.1.008.pdf"
converter = DocumentConverter()
docling_document = converter.convert(source=SOURCE_PATH).document

[INFO] 2026-06-07 22:06:22,666 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-07 22:06:22,678 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-07 22:06:22,682 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-07 22:06:23,636 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-06-07 22:06:23,759 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-07 22:06:23,764 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-07 22:06:24,355 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-07 22:06:24,357 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-07 22:06:24,359 [RapidOCR] download_file.py:68: Init

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [4]:
# Export result as markdown
pdf_markdown = docling_document.export_to_markdown()

In [5]:
# Preview the parsed content
from IPython.display import Markdown, display
display(Markdown(pdf_markdown[:3000]))

NORME MAROCAINE                            NM 10.1.008

Juillet 2007

Béton :Spécifications, performances, production et conformité

## Ce projet de norme a été rédigé par les membres de la commission de rédaction suivants :

- Mr Yahya BOUCHAQOUR  (DAT)
- Mr Khalid BENJELLOUN  (LPEE) :Rapporteur
- Secrétariat :   Mme Bouchra AMMARI  (DRCR) Mme Imane JABRI  (DAT)
- Mr Abdelfattah MOBARAA (DRCR)
- Mr Adil CHENNOUF  (HOLCIM)
- Mr Ahmed FELLAKI  (BETOMAR)
- Mr Azelaarab ALAOUI  (ONCF)
- Mr Mohamed BOUFOUS  (Ciments du Maroc)
- Mr Mohamed TOUGUY  (Armabéton)
- Mme Nezha LABIED  (DEP)
- Mr Othmane BANSATOR (LAFARGE-béton)

## Sommaire

| Introduction                                                                                                           | ...............................................................................................................................9   |
|------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------|
| 1 domaine d'application                                                                                                | ..........................................................................................................9                        |
| 2 références normatives                                                                                                | ........................................................................................................10                         |
| 3 définitions symboles et abréviations                                                                                 | ................................................................................11                                                 |
| 3.1 termes et définitions                                                                                              | ...................................................................................................11                              |
| 3.1.1 béton                                                                                                            | ......................................................................................................................11           |
| 3.1.2 béton frais                                                                                                      | .............................................................................................................11                    |
| 3.1.3 béton durci                                                                                                      | ............................................................................................................11                     |
| 3.1.4 béton de chantier                                

In [6]:
# Save result to file for later use by Ragas loader
with open("nm_10_1_008.md", "w", encoding="utf-8") as md_file:
    md_file.write(pdf_markdown)

# Document Chunking

> The NM 10.1.008 standard is structured with numbered sections, tables, and annexes. A hybrid chunking strategy respects the document structure while keeping chunks small enough to be precise. We use `sentence-transformers/all-MiniLM-L12-v2` as the tokenizer since it handles French text adequately and is compatible with the embedding model.

In [7]:
# Hybrid chunking requires a tokenizer
from transformers import AutoTokenizer
TOKENIZER_MODEL_ID = "sentence-transformers/all-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL_ID)
# test tokenizer on a sample French phrase from the norm
ids = tokenizer.encode("Béton à haute résistance, classe B50")
print(ids)
tokens = tokenizer.convert_ids_to_tokens(ids, skip_special_tokens=True)
print(tokens)

[101, 6655, 2239, 1037, 18535, 5012, 1010, 2465, 2063, 1038, 12376, 102]
['bet', '##on', 'a', 'haute', 'resistance', ',', 'class', '##e', 'b', '##50']


In [8]:
from docling.chunking import HybridChunker
# Max tokens per chunk: kept at 64 for precise retrieval over this normative text
MAX_TOKENS = 64
chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
    merge_peers=True
)

In [9]:
# Apply chunker on the NM 10.1.008 document
chunks = list(chunker.chunk(docling_document))
chunks = [chunk.text for chunk in chunks]
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:4]):
    print(f"------- chunk_{i+1} -------")
    print(chunk)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4945 > 512). Running this sequence through the model will result in indexing errors


Number of chunks: 1464
------- chunk_1 -------
NORME MAROCAINE                            NM 10.1.008
Juillet 2007
Béton :Spécifications, performances, production et conformité
------- chunk_2 -------
- Mr Yahya BOUCHAQOUR  (DAT)
- Mr Khalid BENJELLOUN  (LPEE) :Rapporteur
------- chunk_3 -------
- Secrétariat :   Mme Bouchra AMMARI  (DRCR) Mme Imane JABRI  (DAT)
------- chunk_4 -------
- Mr Abdelfattah MOBARAA (DRCR)
- Mr Adil CHENNOUF  (HOLCIM)
- Mr Ahmed FELLAKI  (BETOMAR)


# Chunk Embedding

> We embed all chunks using `sentence-transformers/all-MiniLM-L12-v2`. This model produces 384-dimensional vectors and handles French normative text at reasonable quality. For a production system targeting French/Arabic, consider upgrading to `intfloat/multilingual-e5-large`.

In [10]:
from sentence_transformers import SentenceTransformer
EMBEDDING_MODEL_ID = TOKENIZER_MODEL_ID
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# Test the embedding model on concrete norm terminology
queries = ["classe de resistance B25", "rapport eau/ciment"]
embeddings = embedding_model.encode(queries)
print(f"Embedding dim: {embeddings.shape}")
similarity = embedding_model.similarity(embeddings[0], embeddings[1]).squeeze().item()
print(f"Similarity between the two queries: {similarity:.4f}")

Embedding dim: (2, 384)
Similarity between the two queries: 0.0317


In [12]:
# Embed all chunks from the NM 10.1.008 document
chunks_embedding = embedding_model.encode(chunks)
print(f"Chunks embedding shape: {chunks_embedding.shape}")

Chunks embedding shape: (1464, 384)


# Store in VectorDB / Index

> We use FAISS (in-memory) as the vector index. This is suitable for a single-document RAG over NM 10.1.008. In a multi-norm production system, consider PGVector or Qdrant for persistence and filtering across multiple Moroccan standards (NM 10.1.004, NM 10.1.271, etc.).

In [13]:
import faiss
import numpy as np
# 1. define embedding dimension
dim = chunks_embedding.shape[-1]
# 2. create L2 index
index = faiss.IndexFlatL2(dim)
# 3. add chunk embeddings to index
index.add(chunks_embedding)

In [14]:
# 4. test search on a concrete norm question
query = ["quelles sont les classes de resistance a la compression"]
query_embedding = embedding_model.encode(query)
top_k = 4
distance_scores, indices = index.search(query_embedding, top_k)
print(f"top_k indices: {indices[0]}")
print(f"L2 distance scores: {distance_scores[0]}")

top_k indices: [ 459 1285  555  839]
L2 distance scores: [0.3883927  0.3971764  0.4206127  0.42986974]


In [15]:
# Map indices back to chunks
chunks_array = np.array(chunks)
retrieved_chunks = chunks_array[indices[-1]].tolist()
for i, (rc, dist) in enumerate(zip(retrieved_chunks, distance_scores[0])):
    print(f"------- chunk_{i+1} -----------")
    print(f"Distance score: {dist:.4f}")
    print(f"Chunk: {rc}")

------- chunk_1 -----------
Distance score: 0.3884
Chunk: - Blg.../...., Classes de résistance à la compression dans le cas des bétons légers
------- chunk_2 -----------
Distance score: 0.3972
Chunk: - -référence à la norme NM 10.1.008;
- -classe de résistance à la compression : classe de résistance telle que définie au Tableaux 7 ou 8, par exemple B 25 ;
------- chunk_3 -----------
Distance score: 0.4206
Chunk: Lorsque le béton est classé selon sa résistance à la compression, le Tableau 7
------- chunk_4 -----------
Distance score: 0.4299
Chunk: ) et la résistance moyenne à la compression à 28 jours (f cm,28  ) '


# Reranker

* Reranker model: https://huggingface.co/jinaai/jina-reranker-v3

> The reranker re-scores retrieved chunks by relevance to the query. This is critical for normative text where multiple sections may use similar terminology (e.g., multiple clauses mentioning `rapport eau/ciment`) but only one is directly responsive to the question.

In [16]:
from transformers import AutoModel
RERANKER_MODEL_ID = 'jinaai/jina-reranker-v3'
reranker_model = AutoModel.from_pretrained(
    RERANKER_MODEL_ID,
    dtype="auto",
    device_map="auto",
    trust_remote_code=True,
).eval()

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v3:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

In [17]:
# Rerank retrieved chunks and select top_n=2
reranker_result = reranker_model.rerank(query[0], retrieved_chunks, top_n=2)
for result in reranker_result:
    print("--------")
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document']}")

--------
Score: 0.2441
Document: - -référence à la norme NM 10.1.008;
- -classe de résistance à la compression : classe de résistance telle que définie au Tableaux 7 ou 8, par exemple B 25 ;
--------
Score: 0.1798
Document: - Blg.../...., Classes de résistance à la compression dans le cas des bétons légers


# LLM Augmentation

* Groq API keys: https://console.groq.com/keys

> We use the Groq inference API with the OpenAI-compatible SDK. The model answers questions about NM 10.1.008 using only the retrieved context, ensuring answers stay grounded in the normative document.

In [18]:
from openai import OpenAI
BASE_URL = "https://api.groq.com/openai/v1"
API_KEY = "*************"  # replace with your Groq API key
MODEL = "llama-3.3-70b-versatile"  # change to any available Groq model
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

In [19]:
# Extract document text from reranker results
context = [doc["document"] for doc in reranker_result]
context

['- -référence à la norme NM 10.1.008;\n- -classe de résistance à la compression : classe de résistance telle que définie au Tableaux 7 ou 8, par exemple B 25 ;',
 '- Blg.../...., Classes de résistance à la compression dans le cas des bétons légers']

In [20]:
# Prepare prompt template for NM 10.1.008 questions
user_query = input("Posez une question sur la norme NM 10.1.008: ")
prompt = f"""
Repondez a la question en utilisant UNIQUEMENT le contexte ci-dessous extrait de la norme marocaine NM 10.1.008.

Contexte:
{context}

Question:
{user_query}

Si la reponse ne se trouve pas dans le contexte, dites que vous ne savez pas.
"""

Posez une question sur la norme NM 10.1.008: what present this file


In [21]:
# Generate response
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)
response.choices[0].message.content

'Il semble que ce fichier présente des informations sur les classes de résistance à la compression des bétons, notamment selon la norme NM 10.1.008, avec des références à des tableaux spécifiques (Tableaux 7 ou 8) et des exemples de classes de résistance (comme B 25).'

In [22]:
def apply_chat_template(context, query):
    prompt = f"""
    Repondez a la question en utilisant UNIQUEMENT le contexte ci-dessous extrait de la norme marocaine NM 10.1.008.

    Contexte:
    {context}

    Question:
    {query}

    Si la reponse ne se trouve pas dans le contexte, dites que vous ne savez pas.
    """
    return prompt


def chat_document(question, index_top_k=4, reranker_top_n=2):
    # 1. query embedding
    query_embedding = embedding_model.encode([question])
    # 2. index search
    sim, indices = index.search(query_embedding, k=index_top_k)
    # 3. indices to chunks
    selected_chunks = chunks_array[indices].tolist()[0]
    # 4. reranker
    results = reranker_model.rerank(question, selected_chunks, top_n=reranker_top_n)
    context = [r["document"] for r in results]
    # 5. apply chat template
    prompt = apply_chat_template(context, question)
    # 6. generate answer
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content, context

In [41]:
# Test with a question relevant to the norm
chat_document("Quelle est la resistance caracteristique minimale sur cylindres pour la classe B25?")

('La résistance caractéristique minimale sur cylindres pour la classe B25 est de 20 N/mm² (MPa).',
 ['\n sur cylindres f ck-cyl N/mm 2 (MPA) = 20. B20, Résistance caractéristique minimale sur cubes f ck-cube N/mm 2 (MPA) = 25. B25, Résistance caractéristique minimale sur cylindres f',
  '\n Résistance caractéristique minimale sur cylindres f ck-cyl N/mm 2 (MPA) = 90. B90, Résistance caractéristique minimale sur cubes f ck-cube N/mm 2 (MPA) = 105. B100, Résistance caractéristique'])